# P-median Problem Example with Pyomo

cuOpt is NVIDIA's GPU accelerated solver that delivers massive speedups for real-world LP, MIP, and VRP workloads.

cuOpt seemlessly integrates with modeling languages. You can drop cuOpt into existing models built with pyomo, PuLP, cvxpy and AMPL, with minimal refactoring. Let's take a look at an example solving a simple MIP problem with cuOpt.

To run this in Google Colab, download the notebook and upload it to Google Colab. Make sure you are running this on a T4 GPU.

If you are running this in the cuOpt container, you are good to go!


## 1. Install Dependencies

To make sure we are good to go, let's install Pyomo and cuOpt.

__[Pyomo](https://github.com/Pyomo/pyomo)__ is a popular linear and mixed integer programming modeler written in Python.


If you are running this notebook in Google Colab, or elsewhere outside the container where cuOpt is not yet installed, uncomment the pip install command to install cuOpt.

In [ ]:
import subprocess
import html
from IPython.display import display, HTML

def check_gpu():
    try:
        result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=5)
        result.check_returncode()
        lines = result.stdout.splitlines()
        gpu_info = lines[2] if len(lines) > 2 else "GPU detected"
        gpu_info_escaped = html.escape(gpu_info)
        display(HTML(f"""
        <div style="border:2px solid #4CAF50;padding:10px;border-radius:10px;background:#e8f5e9;">
            <h3>✅ GPU is enabled</h3>
            <pre>{gpu_info_escaped}</pre>
        </div>
        """))
        return True
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired, FileNotFoundError, IndexError) as e:
        display(HTML("""
        <div style="border:2px solid red;padding:15px;border-radius:10px;background:#ffeeee;">
            <h3>⚠️ GPU not detected!</h3>
            <p>This notebook requires a <b>GPU runtime</b>.</p>
            
            <h4>If running in Google Colab:</h4>
            <ol>
              <li>Click on <b>Runtime → Change runtime type</b></li>
              <li>Set <b>Hardware accelerator</b> to <b>GPU</b></li>
              <li>Then click <b>Save</b> and <b>Runtime → Restart runtime</b>.</li>
            </ol>
            
            <h4>If running in Docker:</h4>
            <ol>
              <li>Ensure you have <b>NVIDIA Docker runtime</b> installed (<code>nvidia-docker2</code>)</li>
              <li>Run container with GPU support: <code>docker run --gpus all ...</code></li>
              <li>Or use: <code>docker run --runtime=nvidia ...</code> for older Docker versions</li>
              <li>Verify GPU access: <code>docker run --gpus all nvidia/cuda:12.0.0-base-ubuntu22.04 nvidia-smi</code></li>
            </ol>
            
            <p><b>Additional resources:</b></p>
            <ul>
              <li><a href="https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/install-guide.html" target="_blank">NVIDIA Container Toolkit Installation Guide</a></li>
            </ul>
        </div>
        """))
        return False

check_gpu()

In [ ]:
!pip install pyomo==6.10.0

In [ ]:
# Enable this in case you are running this in google colab or such places where cuOpt is not yet installed
#!pip uninstall -y cuda-python cuda-bindings cuda-core
#!pip install --upgrade --extra-index-url=https://pypi.nvidia.com cuopt-cu12 nvidia-nvjitlink-cu12 rapids-logger==0.1.19
#!pip install --upgrade --extra-index-url=https://pypi.nvidia.com cuopt-cu13 nvidia-nvjitlink-cu13 rapids-logger==0.1.19

## 2. Problem Setup

A city wants to place p = 6 ambulance stations to serve 50 demand zones (neighborhood centroids). Every zone must be assigned to exactly one open station, and we want to minimize total (weighted) travel distance.

This is the classic p-median: choose facility locations + assign demand points to them.

- We have N=50 demand points (clients)

- M=20 candidate facility sites

- Open exactly p=6 sites

- Assign every client to one open site

- Objective: Minimize total weighted distance

In [ ]:
# Generate problem parameters

import random, math
random.seed(7)

# sizes
N = 50   # clients
M = 20   # candidate sites
p = 6    # number of facilities to open

clients = list(range(N))
sites = list(range(M))

# coordinates
client_xy = {i: (random.uniform(0, 20), random.uniform(0, 20)) for i in clients}
site_xy   = {j: (random.uniform(0, 20), random.uniform(0, 20)) for j in sites}

# demand weights (e.g., expected calls)
w = {i: random.randint(1, 10) for i in clients}

def euclid(a, b):
    return math.hypot(a[0] - b[0], a[1] - b[1])

# distance matrix d[i,j]
d = {(i, j): euclid(client_xy[i], site_xy[j]) for i in clients for j in sites}

print("Example weights:", list(w.items())[:5])
print("Example distance d[0,0]:", d[(0,0)])

### Decision variables

- y<sub>j</sub> ∈ {0,1} : open station at site j

- x<sub>ij</sub> ∈ {0,1} : assign client i to site j


### Objective
Minimize weighted distance: 

&emsp; min⁡ ∑<sub>i∈I</sub> ∑<sub>j∈J</sub> w<sub>ij</sub> d<sub>ij</sub> x<sub>ij</sub>


### Constraints
1. Every client assigned to exactly one site:
  ∑<sub>j</sub> x<sub>ij</sub> = 1 ∀i
   
2. Assign only to opened sites:
  x<sub>ij</sub> ≤ y<sub>j</sub> ∀i,j

3. Open exactly p sites:
  ∑<sub>j</sub> y<sub>j</sub> = p

In [ ]:
# Model the problem
import pyomo.environ as pyo

m = pyo.ConcreteModel()

# Sets
m.I = pyo.Set(initialize=clients)  # clients
m.J = pyo.Set(initialize=sites)    # sites

# Params
m.w = pyo.Param(m.I, initialize=w, within=pyo.PositiveIntegers)
m.d = pyo.Param(m.I, m.J, initialize=d, within=pyo.NonNegativeReals)
m.p = pyo.Param(initialize=p)

# Decision variables
m.x = pyo.Var(m.I, m.J, domain=pyo.Binary)  # assignment
m.y = pyo.Var(m.J, domain=pyo.Binary)       # open facility

# Objective
def obj_rule(m):
    return sum(m.w[i] * m.d[i, j] * m.x[i, j] for i in m.I for j in m.J)
m.obj = pyo.Objective(rule=obj_rule, sense=pyo.minimize)

# Constraints
def assign_once_rule(m, i):
    return sum(m.x[i, j] for j in m.J) == 1
m.assign_once = pyo.Constraint(m.I, rule=assign_once_rule)

def open_link_rule(m, i, j):
    return m.x[i, j] <= m.y[j]
m.open_link = pyo.Constraint(m.I, m.J, rule=open_link_rule)

def open_p_rule(m):
    return sum(m.y[j] for j in m.J) == m.p
m.open_p = pyo.Constraint(rule=open_p_rule)

## 3. Problem Solution

Pyomo calls on the cuOpt solver, which finds the optimal site locations. 

In [ ]:
# Solve the problem using CUOPT
solver = pyo.SolverFactory("cuopt")
results = solver.solve(m)

In [ ]:
%matplotlib inline

# Print summary and plot results

def print_summary(m):
    print("\n=============== Summary ===============")
    # opened sites
    opened = [j for j in m.J if pyo.value(m.y[j]) > 0.5]
    print("Opened sites:", opened)
    
    # assignment per client
    assign = {}
    for i in m.I:
        for j in m.J:
            if pyo.value(m.x[i, j]) > 0.5:
                assign[i] = j
                break
    
    obj_val = pyo.value(m.obj)
    total_weight = sum(w.values())
    wavg_dist = sum(w[i] * d[(i, assign[i])] for i in clients) / total_weight
    max_dist = max(d[(i, assign[i])] for i in clients)
    
    print(f"Objective (weighted distance): {obj_val:.4f}")
    print(f"Weighted avg distance: {wavg_dist:.4f}")
    print(f"Max distance: {max_dist:.4f}")
    return opened, assign

def plot_result(opened, assign):
    import matplotlib.pyplot as plt
    
    plt.figure()
    
    # clients (size by weight)
    cx = [client_xy[i][0] for i in clients]
    cy = [client_xy[i][1] for i in clients]
    cs = [25 + 10*w[i] for i in clients]
    plt.scatter(cx, cy, s=cs, marker="o", label="Clients")
    
    # all candidate sites
    sx = [site_xy[j][0] for j in sites]
    sy = [site_xy[j][1] for j in sites]
    plt.scatter(sx, sy, s=60, marker="^", label="Candidate sites")
    
    # opened sites
    ox = [site_xy[j][0] for j in opened]
    oy = [site_xy[j][1] for j in opened]
    plt.scatter(ox, oy, s=200, marker="^", label="Opened", edgecolors="black")
    
    # assignment lines
    for i in clients:
        j = assign[i]
        plt.plot([client_xy[i][0], site_xy[j][0]],
                 [client_xy[i][1], site_xy[j][1]],
                 linewidth=0.5)
    
    plt.title("Pyomo p-Median: assignments to opened sites")
    plt.legend()
    plt.show()

opened, assign = print_summary(m)
plot_result(opened, assign)

## 4. Extension: Capacitated p-median

If each station can handle at most K total demand weight:

&emsp; ∑<sub>i</sub>w<sub>i</sub>x<sub>ij</sub> ≤ Ky<sub>j</sub> ∀j

In [ ]:
K = 50

# Add Constraint
def cap_rule(m, j):
    return sum(m.w[i] * m.x[i, j] for i in m.I) <= K * m.y[j]
m.capacity = pyo.Constraint(m.J, rule=cap_rule)

# Re-optimize
result = solver.solve(m)

In [ ]:
%matplotlib inline
# Print summary and plot results
opened, assign = print_summary(m)
plot_result(opened, assign)